In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_beneficiarios"
TARGET_TABLE = "homologacao.salutar_saude.silver_beneficiarios"

DATE_COLS = ["data_nascimento", "data_adesao", "data_cancelamento"]

run_ts = datetime.now()

print(f"Pipeline  : silver_beneficiarios")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_beneficiarios
Iniciado  : 2026-07-04 18:10:25
Origem    : homologacao.salutar_saude.raw_beneficiarios
Destino   : homologacao.salutar_saude.silver_beneficiarios


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 1,558 registros lidos de homologacao.salutar_saude.raw_beneficiarios


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_sexo(col_name: str) -> F.Column:
    """Normaliza sexo: M/Masculino → Masculino, F/Feminino → Feminino."""
    return (
        F.when(F.col(col_name).isin("M", "Masculino"), "Masculino")
         .when(F.col(col_name).isin("F", "Feminino"), "Feminino")
         .otherwise(F.lit(None))
         .alias(col_name)
    )


def transform(df: DataFrame) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.
    """
    cols = [c for c in df.columns if c != "_rescued_data"]
    return (
        df.select(
            *[
                parse_date(c) if c in DATE_COLS
                else parse_sexo(c) if c == "sexo"
                else F.col(c)
                for c in cols
            ]
        )
        .withColumn("_updated_at", F.lit(run_ts).cast("timestamp"))
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
df_silver = transform(df_raw)

# ── Validação de qualidade ──────────────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + ["sexo"]

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
)
unexpected_sexo = df_silver.filter(
    F.col("sexo").isNotNull() & ~F.col("sexo").isin("Masculino", "Feminino")
).count()
n_silver = df_silver.count()

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros            : {n_silver:,}")
print(f"  Correspondência com RAW       : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_sexo else "[OK]  "
print(f"  {tag} sexo valores inesperados  : {unexpected_sexo:,}")
print("─" * 65)

assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
assert unexpected_sexo == 0, f"{unexpected_sexo} valores inesperados em 'sexo'"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros            : 1,558
  Correspondência com RAW       : [OK]
  [OK]   data_nascimento: 0 nulos
  [OK]   data_adesao: 0 nulos
  [WARN] data_cancelamento: 1,383 nulos
  [WARN] sexo: 298 nulos
  [OK]   sexo valores inesperados  : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Chave de merge: beneficiario_id (chave primária da tabela)
MERGE_KEY = "beneficiario_id"

# Deduplica pela chave de merge antes do MERGE
# (Delta exige cardinalidade 1:1 na fonte — múltiplas linhas com a mesma chave causam erro)
df_to_merge = df_silver.dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza registro existente se houver mudança
        .whenNotMatchedInsertAll()    # insere novo beneficiário
        # .whenNotMatchedBySourceDelete()  # descomente para remover registros deletados da RAW
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_silver.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] MERGE concluído      : homologacao.salutar_saude.silver_beneficiarios
[OK] Registros na tabela  : 1,558
[OK] Fim do pipeline      : 2026-07-04 18:14:37


In [0]:
%sql
SELECT *
FROM homologacao.salutar_saude.silver_beneficiarios

beneficiario_id,contrato_id,empresa_id,data_nascimento,sexo,tipo_beneficiario,data_adesao,data_cancelamento,_updated_at
1,1,1,1968-09-23,Masculino,Titular,2024-09-22,null,2026-07-04T18:10:25.787Z
2,1,1,2001-07-21,Masculino,Dependente,2024-09-13,null,2026-07-04T18:10:25.787Z
3,1,1,2011-01-27,Masculino,Dependente,2024-08-23,null,2026-07-04T18:10:25.787Z
4,1,1,1968-10-31,Feminino,Dependente,2024-11-12,null,2026-07-04T18:10:25.787Z
5,1,1,1958-09-11,Masculino,Titular,2024-11-02,null,2026-07-04T18:10:25.787Z
6,1,1,2006-04-30,Feminino,Dependente,2024-08-15,null,2026-07-04T18:10:25.787Z
7,1,1,1982-02-07,Feminino,Titular,2024-11-23,null,2026-07-04T18:10:25.787Z
8,1,1,1960-05-17,Feminino,Titular,2024-10-12,null,2026-07-04T18:10:25.787Z
9,1,1,1983-08-25,Masculino,Dependente,2024-08-28,null,2026-07-04T18:10:25.787Z
10,1,1,1976-09-08,Feminino,Titular,2024-10-08,null,2026-07-04T18:10:25.787Z
